# 00 · Data checks (exploratory split)

**Business question:** can we trust these 4,873 headline tests before reading any result?

Checks: what one test looks like, whether readers were split evenly (SRM), how big the tests were (power), and how stories are tagged by topic. Only the exploratory split is used; the confirmatory split stays locked until the pre-registration is committed.

In [ ]:
import matplotlib.pyplot as plt

from headline_lab.checks import srm_by_test
from headline_lab.data import load_packages, summarise_tests
from headline_lab.power import mde_by_test
from headline_lab.topics import tag_tests

packages = load_packages("../data/raw")  # exploratory only
tests = summarise_tests(packages)
len(packages), len(tests)

## 1. What one test looks like
One story, several headlines, readers split at random.

In [ ]:
example = tests.query("arms >= 3").iloc[0]["clickability_test_id"]
packages.query("clickability_test_id == @example")[["headline", "impressions", "clicks", "ctr"]]

In [ ]:
tests[["arms", "impressions", "ctr"]].describe().round(4)

## 2. Were readers split evenly? (SRM per test)

In [ ]:
srm = srm_by_test(packages)
srm["has_mismatch"].mean().round(4), srm["has_mismatch"].sum()

**Takeaway:** TODO (Kaan). What share of tests failed, and what should we do with them?

## 3. Were the tests big enough? (power)
Smallest change in click rate each test could reliably detect, as a share of its click rate.

In [ ]:
power = mde_by_test(tests)
power["mde_relative"].describe(percentiles=[0.1, 0.5, 0.9]).round(3)

In [ ]:
# most headline changes move clicks by a few percent, so look at the share that could see 10%
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(power["mde_relative"].clip(upper=1) * 100, bins=50, color="#2a78d6")
ax.axvline(10, color="#eb6834", lw=2, label="a 10% change")
ax.set_title("Most headline tests could only detect large changes")
ax.set_xlabel("Smallest detectable change in click rate (%, capped at 100)")
ax.set_ylabel("Tests")
ax.legend()
plt.show()
(power["mde_relative"] <= 0.10).mean().round(3)

**Takeaway:** TODO (Kaan). What share of tests could detect a 10% change?

## 4. Topics

In [ ]:
topics = tag_tests(packages)
topics.value_counts(), topics.value_counts(normalize=True).round(3)

In [ ]:
# a few of each, to sanity-check the tagger by eye
sample = packages.groupby("clickability_test_id").first().join(topics)
for name in ["politics", "business"]:
    print(name.upper())
    print(sample.query("topic == @name")["headline"].sample(8, random_state=1).to_string(), "\n")

**Next:** run `headline-lab-label-sample` in the terminal, label the 200 stories in `data/labels/to_label.csv`, then check the tagger:

```python
from headline_lab.labels import validate
validate(pd.read_csv("../data/labels/to_label.csv"), topics)
```